In [1]:
import numpy as np

def pkcs7_padding(data):
    # Applying PKCS#7 padding
    padding_length = 8 - (len(data) % 8)
    if padding_length == 0:
        padding_length = 8
    return data + bytes([padding_length]) * padding_length

def bytes_to_bits(data):
    return np.unpackbits(np.frombuffer(data, dtype=np.uint8))


In [2]:
IP = [
    58,50,42,34,26,18,10,2,60,52,44,36,28,20,12,4,
    62,54,46,38,30,22,14,6,64,56,48,40,32,24,16,8,
    57,49,41,33,25,17,9,1,59,51,43,35,27,19,11,3,
    61,53,45,37,29,21,13,5,63,55,47,39,31,23,15,7
]

def permute(bits, table):
    # Applies a permutation table
    return bits[np.array(table) - 1]

In [3]:
PC1 = [
    57,49,41,33,25,17,9,
    1,58,50,42,34,26,18,
    10,2,59,51,43,35,27,
    19,11,3,60,52,44,36,
    63,55,47,39,31,23,15,
    7,62,54,46,38,30,22,
    14,6,61,53,45,37,29,
    21,13,5,28,20,12,4
]

PC2 = [
    14,17,11,24,1,5,3,28,15,6,21,10,
    23,19,12,4,26,8,16,7,27,20,13,2,
    41,52,31,37,47,55,30,40,51,45,33,48,
    44,49,39,56,34,53,46,42,50,36,29,32
]

SHIFTS = [1, 1, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 1]

def left_shift(bits, n):
    # Left circular shift by n bits using NumPy roll
    return np.roll(bits, -n)

def determine_16_subkeys(key):
    # Converts the initial key from bytes to bits
    key_bits = np.unpackbits(np.frombuffer(key, dtype=np.uint8))

    # Does the initial key permutation. The input is a 64 bit key and the output is a 56 bit key
    key_bits_perm = permute(key_bits, PC1)

    # Splits the 56 key bits is half
    C, D = key_bits_perm[:28], key_bits_perm[28:]
    
    # Generated the 16 subkeys
    subkeys = []
    for shift in SHIFTS:
        C = left_shift(C, shift)
        D = left_shift(D, shift)

        # Concatenates the C and D Numpy arrays to get 56 bits array
        CD = np.concatenate((C, D))

        # Does the second permutation. The input is a 56 bit key and the output is a 48 bit key
        perm_CD = permute(CD, PC2)

        subkeys.append(perm_CD)
    return subkeys

        

In [4]:
EXPANSION_TABLE = [
    32,1,2,3,4,5,4,5,6,7,8,9,8,9,10,11,12,13,
    12,13,14,15,16,17,16,17,18,19,20,21,20,21,22,23,24,25,
    24,25,26,27,28,29,28,29,30,31,32,1
]

SBOXES = [
    # S1
    np.array([
        [14,4,13,1,2,15,11,8,3,10,6,12,5,9,0,7],
        [0,15,7,4,14,2,13,1,10,6,12,11,9,5,3,8],
        [4,1,14,8,13,6,2,11,15,12,9,7,3,10,5,0],
        [15,12,8,2,4,9,1,7,5,11,3,14,10,0,6,13]
    ]),
    # S2
    np.array([
        [15,1,8,14,6,11,3,4,9,7,2,13,12,0,5,10],
        [3,13,4,7,15,2,8,14,12,0,1,10,6,9,11,5],
        [0,14,7,11,10,4,13,1,5,8,12,6,9,3,2,15],
        [13,8,10,1,3,15,4,2,11,6,7,12,0,5,14,9]
    ]),
    # S3
    np.array([
        [10,0,9,14,6,3,15,5,1,13,12,7,11,4,2,8],
        [13,7,0,9,3,4,6,10,2,8,5,14,12,11,15,1],
        [13,6,4,9,8,15,3,0,11,1,2,12,5,10,14,7],
        [1,10,13,0,6,9,8,7,4,15,14,3,11,5,2,12]
    ]),
    # S4
    np.array([
        [7,13,14,3,0,6,9,10,1,2,8,5,11,12,4,15],
        [13,8,11,5,6,15,0,3,4,7,2,12,1,10,14,9],
        [10,6,9,0,12,11,7,13,15,1,3,14,5,2,8,4],
        [3,15,0,6,10,1,13,8,9,4,5,11,12,7,2,14]
    ]),
    # S5
    np.array([
        [2,12,4,1,7,10,11,6,8,5,3,15,13,0,14,9],
        [14,11,2,12,4,7,13,1,5,0,15,10,3,9,8,6],
        [4,2,1,11,10,13,7,8,15,9,12,5,6,3,0,14],
        [11,8,12,7,1,14,2,13,6,15,0,9,10,4,5,3]
    ]),
    # S6
    np.array([
        [12,1,10,15,9,2,6,8,0,13,3,4,14,7,5,11],
        [10,15,4,2,7,12,9,5,6,1,13,14,0,11,3,8],
        [9,14,15,5,2,8,12,3,7,0,4,10,1,13,11,6],
        [4,3,2,12,9,5,15,10,11,14,1,7,6,0,8,13]
    ]),
    # S7
    np.array([
        [4,11,2,14,15,0,8,13,3,12,9,7,5,10,6,1],
        [13,0,11,7,4,9,1,10,14,3,5,12,6,8,2,15],
        [1,4,11,13,12,3,7,14,10,15,6,8,0,5,9,2],
        [6,11,13,8,1,4,10,7,9,5,0,15,14,2,3,12]
    ]),
    # S8
    np.array([
        [13,2,8,4,6,15,11,1,10,9,3,14,5,0,12,7],
        [1,15,13,8,10,3,7,4,12,5,6,11,0,14,9,2],
        [7,11,4,1,9,12,14,2,0,6,10,13,15,3,5,8],
        [2,1,14,7,4,10,8,13,15,12,9,0,3,5,6,11]
    ])
]

P_TABLE = [
    16,7,20,21,29,12,28,17,1,15,23,26,5,18,31,10,
    2,8,24,14,32,27,3,9,19,13,30,6,22,11,4,25
]

FP = [
    40,8,48,16,56,24,64,32,39,7,47,15,55,23,63,31,
    38,6,46,14,54,22,62,30,37,5,45,13,53,21,61,29,
    36,4,44,12,52,20,60,28,35,3,43,11,51,19,59,27,
    34,2,42,10,50,18,58,26,33,1,41,9,49,17,57,25
]

def sbox_substitution(bits):
    """
    Applies the 8 DES S-boxes to a 48-bit input.
    Each S-box takes 6 bits → outputs 4 bits.
    Result: 32-bit NumPy array.
    """
    output_bits = []

    # There are 8 S-boxes, each takes 6 bits
    for i in range(8):
        # Get 6 bits for this S-box
        six_bits = bits[i * 6 : (i + 1) * 6]

        # Outer bits (1st and 6th) form the row
        row = (six_bits[0] << 1) | six_bits[5]

        # Middle 4 bits form the column
        col = (six_bits[1] << 3) | (six_bits[2] << 2) | (six_bits[3] << 1) | six_bits[4]

        # Look up the value from the corresponding S-box
        sbox_value = SBOXES[i][row, col]  # a number between 0 and 15

        # Convert that value into 4 bits
        four_bits = [
            (sbox_value >> 3) & 1,  # bit 1
            (sbox_value >> 2) & 1,  # bit 2
            (sbox_value >> 1) & 1,  # bit 3
            sbox_value & 1          # bit 4
        ]

        # Add those 4 bits to our final output
        output_bits.extend(four_bits)

        # print details for learning
        #print(f"S{i+1}: input={six_bits.tolist()}, row={row}, col={col}, "
        #      f"value={sbox_value:02d}, bits={four_bits}")

    # Return as NumPy array of 0s and 1s
    return np.array(output_bits, dtype=np.uint8)

def feistel_round(R, ki):
    # Using the Expansion table, we duplicate some bits from R to make it longer. From 32 bits, we get to 48 to match the key length
    R_expanded = permute(R, EXPANSION_TABLE)

    # We do the XOR operation between the Expanded R and the key
    x = np.bitwise_xor(R_expanded, ki)

    # Devide x into 8 groups of 6-bit chunks and pass each 6-bit chunk through its S-box. We get from a 48 bit array to a 32 bit array
    s = sbox_substitution(x)

    # After all 8 S-boxes, the 32 bits are rearranged using the P-table
    s_perm = permute(s, P_TABLE)
    return s_perm

def bits_to_bytes(bits):
    # Converts a NumPy bit array back to bytes
    return np.packbits(bits).tobytes()

def DES_block_encrypt(block, round_keys):
    # Converts the plaintext to bits
    bits = bytes_to_bits(block)

    # Does the Initial Permutation
    ip_bits = permute(bits, IP)

    # Splits the bits in half: Left and Right
    L, R = ip_bits[:32], ip_bits[32:]

    # Performs 16 Feistel rounds
    for i in range(16):
        feistel_val = feistel_round(R, round_keys[i])
        L, R = R, np.bitwise_xor(L, feistel_val)

    # Concatenated the 2 arrays: Left and Right
    LR = np.concatenate((R, L))

    # Does the Final Permutation
    LR_perm = permute(LR, FP)

    # Converts the output from bits to bytes
    return bits_to_bytes(LR_perm)

def DES_encrypt(plaintext, key):
    round_keys = determine_16_subkeys(key)
    plaintext = pkcs7_padding(plaintext)
    out = bytearray()
    for i in range(0, len(plaintext), 8):
        out.extend(DES_block_encrypt(plaintext[i : i + 8], round_keys))
    return bytes(out)

In [5]:
def DES_block_decrypt(ciphertext_block, round_keys):
    bits = bytes_to_bits(ciphertext_block)
    ip_bits = permute(bits, IP)
    L, R = ip_bits[:32], ip_bits[32:]
    for i in reversed(range(16)):
        feistel_val = feistel_round(R, round_keys[i])
        L, R = R, np.bitwise_xor(L, feistel_val)
    LR = np.concatenate((R, L))
    LR_perm = permute(LR, FP)
    return bits_to_bytes(LR_perm)

def pkcs7_unpadding(data):
    padding_length = data[-1]
    return data[:-padding_length]

def DES_decrypt(ciphertext, key):
    round_keys = determine_16_subkeys(key)
    out = bytearray()
    for i in range(0, len(ciphertext), 8):
        out.extend(DES_block_decrypt(ciphertext[i : i + 8], round_keys))
    return pkcs7_unpadding(bytes(out))

In [6]:
msg = b"Hello everyone!!" # 16 bytes message
key = bytes.fromhex("133457799BBCDFF1") # 8 bytes key
c = DES_encrypt(msg, key)
d = DES_decrypt(c, key)
print(f"Original message: {msg}")
print(f"Decrypted message: {d}")
print("Encryption correct:", d == msg)

Original message: b'Hello everyone!!'
Decrypted message: b'Hello everyone!!'
Encryption correct: True


In [7]:
def DoubleDES_encrypt(plaintext, key1, key2):
    #Performs Double DES encryption:
    # First encryption with key1
    intermediate = DES_encrypt(plaintext, key1)
    # Second encryption with key2
    ciphertext = DES_encrypt(intermediate, key2)
    return ciphertext


def DoubleDES_decrypt(ciphertext, key1, key2):
    #Performs Double DES decryption:
    # First decrypt with key2
    intermediate = DES_decrypt(ciphertext, key2)
    # Then decrypt with key1
    plaintext = DES_decrypt(intermediate, key1)
    return plaintext


In [8]:
msg = b"Hello everyone!!" # 16 bytes message
key1 = bytes.fromhex("133457799BBCDFF1") # 8 bytes key
key2 = bytes.fromhex("387436BACDE83271")
c = DoubleDES_encrypt(msg, key1, key2)
d = DoubleDES_decrypt(c, key1, key2)
print(f"Original message: {msg}")
print(f"Decrypted message: {d}")
print("Encryption correct:", d == msg)

Original message: b'Hello everyone!!'
Decrypted message: b'Hello everyone!!'
Encryption correct: True


In [9]:
from itertools import product

def MITM_attack(plaintext, ciphertext):
    # we try all the possibile values for k1 (2^8 possibilities)
    byte_values = (0x00, 0xFF)
    enc_table = {} # map: first encryption -> k1
    for tup in product(byte_values, repeat = 8):
        key1 = bytes(tup)
        enc = DES_encrypt(plaintext, key1)
        enc_table.setdefault(enc, []).append(key1)

    # we try all the possible values for k2 (2^8 possibilities)
    candidates = []
    for tup in product(byte_values, repeat = 8):
        key2 = bytes(tup)
        dec = DES_decrypt(ciphertext, key2)
        if dec in enc_table:
            for key1 in enc_table[dec]:
                candidates.append((key1, key2))
    return candidates

In [11]:
plaintext = b"Hello!!!" # 8 bytes message
key1 = bytes([0x00,0xFF,0x00,0xFF,0x00,0xFF,0x00,0xFF])
key2 = bytes([0xFF,0xFF,0x00,0x00,0xFF,0xFF,0x00,0x00])
ciphertext = DoubleDES_encrypt(plaintext, key1, key2)
print("Real Key1 and key2:", key1, key2)

candidates = MITM_attack(plaintext, ciphertext)
print(f"Determined keys: {candidates}")

Real Key1 and key2: b'\x00\xff\x00\xff\x00\xff\x00\xff' b'\xff\xff\x00\x00\xff\xff\x00\x00'
Determined keys: [(b'\x00\xff\x00\xff\x00\xff\x00\xff', b'\xff\xff\x00\x00\xff\xff\x00\x00')]
